In [6]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [7]:
data = load_breast_cancer()

X = data.data
y = data.target

print("X shape:", X.shape)
print("Y shape:", y.shape)

X shape: (569, 30)
Y shape: (569,)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

In [9]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
def build_model(hp):
    model = keras.Sequential()

    model.add(layers.Input(shape=(X_train.shape[1],)))
    num_layers = hp.Int(
        "num_layers",
        min_value = 2,
        max_value = 3
    )
    for i in range(num_layers):
        units = hp.Int(f"units_{i}",
                       min_value=10,
                       max_value= 20,
                       step=3)
        model.add(layers.Dense(units=units, activation='relu'))
        drops = hp.Float(
            f"drop_{i}",
            min_value=0.1
            , max_value =0.5
            , step = 0.2
        
        )
        model.add(layers.Dropout(drops))
    model.add(layers.Dense(1, activation='sigmoid'))

    learning_rate = hp.Choice(
        "learning_rate",
        values=[
            0.001,
            0.0005,
            0.0001
        ]
    )
    optimizer_name = hp.Choice(
        "optimizer",
        values=[
            "adam",
            "rmsprop"
        ]
    )

    if optimizer_name == "adam":
        optimizer = keras.optimizers.Adam(
            learning_rate=learning_rate
        )
    else:
        optimizer = keras.optimizers.RMSprop(
            learning_rate=learning_rate
        )

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [19]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=10,
    directory="ann_tuning",
    project_name="breast_cancer_ann"
)

In [20]:
tuner.search(
    X_train,
    y_train,
    epochs=10,
    validation_split=0.2,
    verbose=1
)

Trial 10 Complete [00h 00m 01s]
val_accuracy: 0.6483516693115234

Best val_accuracy So Far: 0.9560439586639404
Total elapsed time: 00h 00m 09s


In [21]:
best_hp = tuner.get_best_hyperparameters(
    num_trials=1
)[0]

In [23]:
print("Best Hyperparameters")

print(
    "Number of hidden layers:",
    best_hp.get("num_layers")
)

print(
    "Learning rate:",
    best_hp.get("learning_rate")
)

print(
    "Optimizer:",
    best_hp.get("optimizer")
)

for i in range(best_hp.get("num_layers")):

    print(
        f"Layer {i+1} neurons:",
        best_hp.get(f"units_{i}")
    )

    print(
        f"Layer {i+1} dropout:",
        best_hp.get(f"drop_{i}")
    )

Best Hyperparameters
Number of hidden layers: 3
Learning rate: 0.001
Optimizer: rmsprop
Layer 1 neurons: 16
Layer 1 dropout: 0.30000000000000004
Layer 2 neurons: 13
Layer 2 dropout: 0.5
Layer 3 neurons: 19
Layer 3 dropout: 0.1


In [24]:
best_model = tuner.get_best_models(
    num_models=1
)[0]

/Users/anzarhussain/Documents/Deep Learning Tensorflow/myenv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(store)


In [25]:
best_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 13)             │           221 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 13)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 19)             │           266 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 19)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            20 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,003 (3.92 KB)

 Trainable params: 1,003 (3.92 KB)

 Non-trainable params: 0 (0.00 B)

In [26]:
history = best_model.fit(
    X_train,
    y_train,
    epochs=20,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8901 - loss: 0.3508 - val_accuracy: 0.9560 - val_loss: 0.2566
Epoch 2/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8984 - loss: 0.3263 - val_accuracy: 0.9560 - val_loss: 0.2255
Epoch 3/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9148 - loss: 0.2878 - val_accuracy: 0.9560 - val_loss: 0.2008
Epoch 4/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9176 - loss: 0.2727 - val_accuracy: 0.9560 - val_loss: 0.1802
Epoch 5/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9176 - loss: 0.2494 - val_accuracy: 0.9560 - val_loss: 0.1623
Epoch 6/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9176 - loss: 0.2351 - val_accuracy: 0.9560 - val_loss: 0.1503
Epoch 7/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9038 - loss: 0.2536 - val_accuracy: 0.9560 - val_loss: 0.1415
Epoch 8/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9203 - loss: 0.2230 - val_accuracy: 0.9560 - val_loss:

In [27]:
test_loss, test_accuracy = best_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

Test Loss: 0.07077420502901077
Test Accuracy: 0.9561403393745422


In [28]:
y_pred_prob = best_model.predict(X_test)

y_pred = (y_pred_prob > 0.5).astype(int)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 


In [29]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.93      0.95      0.94        43
           1       0.97      0.96      0.96        71

    accuracy                           0.96       114
   macro avg       0.95      0.96      0.95       114
weighted avg       0.96      0.96      0.96       114

